# 04b — In-process depth scaling

In-process depth conditions are summarized from passed leaves. N is independent runs, units are microseconds, and evidence is descriptive diagnostic output. Missing depths remain PENDING, never zero.


In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.focused import evidence_label, passed_artifacts, pending_record, percentile_rows
from wafer_analysis.paths import resolve_result_batch

def resolve(experiment, env_name):
    try:
        return resolve_result_batch(experiment, diagnostic_path=os.environ.get(env_name))
    except (FileNotFoundError, RuntimeError, ValueError):
        return None

batch=resolve('e-perf-8','E_PERF_8_DIR'); df=pd.DataFrame() if batch is None else percentile_rows(batch); rows=[]
for depth in ['depth-1','depth-3','depth-5','depth-10']:
    values=df[df.condition==depth] if not df.empty else pd.DataFrame()
    if values.empty: rows.append(pending_record(depth,'no passed percentile leaf','microseconds'))
    else: rows.append({'condition':depth,'status':'READY','N':len(values),'p50_us':values.p50_ns.median()/1e3,'p95_us':values.p95_ns.median()/1e3,'units':'microseconds','uncertainty':'descriptive only','thesis_evidence':False})
out=pd.DataFrame(rows); display(out)
ready=out[out.status=='READY']
if not ready.empty:
    ax=ready.plot(x='condition',y=['p50_us','p95_us'],marker='o'); ax.set_ylabel('Latency (µs)'); ax.set_title('In-process depth scaling — diagnostic')
